# 01 — EDA and Data Cleaning

This notebook loads the Walmart weekly sales data, inspects it, checks for data quality issues, saves a cleaned version, and creates EDA visualizations.

**Dataset:** 45 Walmart stores, weekly sales from Feb 2010 to Oct 2012 (6,435 rows expected).

## 1. Setup — Import libraries

In [ ]:
# Import the libraries we need
import pandas as pd       # For working with tables (DataFrames)
import numpy as np        # For math operations
import matplotlib.pyplot as plt  # For plotting graphs
import seaborn as sns     # For prettier statistical plots
import os                 # For file path operations

# Settings: show all columns, use a clean plot style
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

# Folder to save all figures
FIGURES_DIR = '../report/figures'

## 2. Load the data (Task 1.2)

We read the CSV file and tell pandas the date format is day/month/year (`dayfirst=True`), because `Walmart.csv` uses dates like `05-02-2010` meaning February 5, 2010.

In [ ]:
# Load the raw data from CSV
# dayfirst=True tells pandas the date is in dd-mm-yyyy format
df = pd.read_csv('../data/Walmart.csv', parse_dates=['Date'], dayfirst=True)

# Quick check: print the shape (rows, columns)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")

The dataset has **6,435 rows** and **8 columns**, exactly as expected (45 stores × 143 weeks = 6,435).

### 2.1 Data types

Let's see what type each column is. `Date` should be `datetime64`, numbers should be `float64` or `int64`.

In [ ]:
# Show the data types of each column
print(df.dtypes)

All types look correct: `Store` and `Holiday_Flag` are integers, `Date` is datetime, and the rest are floats.

### 2.2 First few rows

Let's look at the first 5 rows to see what the data looks like.

In [ ]:
# Show the first 5 rows
df.head()

The data is organized by store, then by date within each store. Weekly sales are in the hundred-thousands to millions range.

### 2.3 Basic statistics

`describe()` gives us the count, mean, std, min, max, and quartiles for every numeric column. This is a quick way to spot anything weird.

In [ ]:
# Show basic statistics for all numeric columns
df.describe()

Key observations from `describe()`:
- **Weekly_Sales** ranges from ~$210K to ~$3.8M — big variation across stores.
- **Temperature** ranges from -2°F to 100°F — reasonable for US locations.
- **Holiday_Flag** mean is 0.07, meaning only ~7% of weeks are holiday weeks.
- **CPI** ranges from 126 to 227, and **Unemployment** from 3.9% to 14.3% — both show variation across stores/regions.

### 2.4 Date range and unique stores

We check the earliest and latest dates and how many unique stores there are.

In [ ]:
# Date range
print(f"Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Total weeks covered: {df['Date'].nunique()}")
print(f"Number of unique stores: {df['Store'].nunique()}")
print(f"Store numbers: {sorted(df['Store'].unique())}")

The data covers **Feb 5, 2010 to Oct 26, 2012** — about 2 years and 9 months. There are **143 unique weeks** and **45 stores** (numbered 1 to 45).

---
## 3. Data Cleaning Checks (Task 1.3)

Now we check for common data quality issues: missing values, duplicates, invalid values, and outliers.

### 3.1 Missing values

We check if any column has missing (NaN) values. Missing data can cause models to crash or give wrong results.

In [ ]:
# Count missing values in each column
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

**Finding:** No missing values in any column. The dataset is complete — no imputation needed.

### 3.2 Duplicate rows

We check if any row appears more than once. Each store should have exactly one entry per week.

In [ ]:
# Check for duplicate rows
n_duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {n_duplicates}")

# Also check for duplicate Store+Date pairs (each store-week should be unique)
n_store_date_dupes = df.duplicated(subset=['Store', 'Date']).sum()
print(f"Duplicate Store+Date combinations: {n_store_date_dupes}")

**Finding:** No duplicate rows and no duplicate Store+Date combinations. Each store-week is unique.

### 3.3 Negative or zero sales

Sales should be positive. Negative sales could mean returns or errors. Zero sales might mean a store was closed.

In [ ]:
# Check for negative sales
n_negative = (df['Weekly_Sales'] < 0).sum()
print(f"Rows with negative Weekly_Sales: {n_negative}")

# Check for zero sales
n_zero = (df['Weekly_Sales'] == 0).sum()
print(f"Rows with zero Weekly_Sales: {n_zero}")

**Finding:** No negative or zero sales. All Weekly_Sales values are positive, which is expected for store-level weekly totals.

### 3.4 Rows per store

Each of the 45 stores should have the same number of weeks (143). If a store has fewer rows, some weeks are missing.

In [ ]:
# Count rows per store
rows_per_store = df.groupby('Store').size()

print(f"Expected rows per store: 143")
print(f"Min rows per store: {rows_per_store.min()} (Store {rows_per_store.idxmin()})")
print(f"Max rows per store: {rows_per_store.max()} (Store {rows_per_store.idxmax()})")

# Check if all stores have exactly 143 rows
all_143 = (rows_per_store == 143).all()
print(f"All stores have 143 rows: {all_143}")

**Finding:** All 45 stores have exactly 143 rows each. The data is complete and balanced — no missing weeks for any store.

### 3.5 Outlier check using IQR method

The IQR (Interquartile Range) method flags values that are unusually high or low:
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 − Q1
- Outliers are below Q1 − 1.5×IQR or above Q3 + 1.5×IQR

**Important:** We only *report* outliers here. We do NOT remove them because high sales during holidays are real demand.

In [ ]:
# IQR outlier detection for Weekly_Sales
Q1 = df['Weekly_Sales'].quantile(0.25)
Q3 = df['Weekly_Sales'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Weekly_Sales'] < lower_bound) | (df['Weekly_Sales'] > upper_bound)]

print(f"Q1: ${Q1:,.2f}")
print(f"Q3: ${Q3:,.2f}")
print(f"IQR: ${IQR:,.2f}")
print(f"Lower bound: ${lower_bound:,.2f}")
print(f"Upper bound: ${upper_bound:,.2f}")
print(f"\nNumber of outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}% of data)")
print(f"Outliers below lower bound: {(df['Weekly_Sales'] < lower_bound).sum()}")
print(f"Outliers above upper bound: {(df['Weekly_Sales'] > upper_bound).sum()}")

In [ ]:
# How many of the upper outliers are holiday weeks?
upper_outliers = df[df['Weekly_Sales'] > upper_bound]
holiday_outliers = upper_outliers[upper_outliers['Holiday_Flag'] == 1]
non_holiday_outliers = upper_outliers[upper_outliers['Holiday_Flag'] == 0]

print(f"Upper outliers that are holiday weeks: {len(holiday_outliers)}")
print(f"Upper outliers that are non-holiday weeks: {len(non_holiday_outliers)}")
print(f"\nDecision: Keep all outliers. Holiday spikes are real demand,")
print(f"and large non-holiday sales likely come from bigger stores.")

**Finding:** There are 34 outliers (0.5% of data), all above the upper bound ($2,720,371). Of the 34 upper outliers, 9 are holiday weeks and 25 are non-holiday weeks (likely from the largest stores). **Decision: keep all outliers.**

---
## 4. Save cleaned data (Task 1.4)

Since no rows needed to be removed, the "cleaned" data is the same as the original but with `Date` properly parsed as datetime. We save it to `data/cleaned.csv` as a checkpoint.

In [ ]:
# Save the cleaned dataframe to CSV
# index=False means we don't save the row numbers
df.to_csv('../data/cleaned.csv', index=False)
print(f"Saved cleaned data to data/cleaned.csv ({df.shape[0]} rows, {df.shape[1]} columns)")

---
## 5. EDA Visualizations (Task 1.5)

Now we create 7 graphs to understand the data better. Each graph is saved to `report/figures/`.

### 5.1 Distribution of Weekly_Sales

A histogram and box plot showing how Weekly_Sales values are spread out.

In [ ]:
# Figure 1: Distribution of Weekly Sales — histogram + box plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram shows how often each sales range occurs
axes[0].hist(df['Weekly_Sales'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Weekly Sales ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Weekly Sales')
axes[0].axvline(df['Weekly_Sales'].mean(), color='red', linestyle='--',
                label=f'Mean: ${df["Weekly_Sales"].mean():,.0f}')
axes[0].axvline(df['Weekly_Sales'].median(), color='orange', linestyle='--',
                label=f'Median: ${df["Weekly_Sales"].median():,.0f}')
axes[0].legend()

# Box plot shows median, quartiles, and outliers
axes[1].boxplot(df['Weekly_Sales'], orientation='vertical')
axes[1].set_ylabel('Weekly Sales ($)')
axes[1].set_title('Box Plot of Weekly Sales')

plt.suptitle('Figure 1: Weekly Sales Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_sales_distribution.png'))
plt.show()

**Observation:** The distribution is right-skewed (skewness = 0.67) — the mean ($1.05M) is higher than the median ($961K). Most stores have modest sales, but a few large stores push the tail right. The box plot confirms the upper outliers we found earlier.

### 5.2 Total sales over time (all stores)

A line chart showing the total weekly sales summed across all 45 stores. Red bands mark holiday weeks.

In [ ]:
# Figure 2: Total sales over time
# Sum sales for all stores in each week
weekly_total = df.groupby('Date')['Weekly_Sales'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly_total['Date'], weekly_total['Weekly_Sales'] / 1e6,
        color='steelblue', linewidth=1.2)

# Mark holiday weeks with red vertical bands
holidays = df[df['Holiday_Flag'] == 1][['Date']].drop_duplicates()
for _, row in holidays.iterrows():
    ax.axvline(row['Date'], color='red', alpha=0.15, linewidth=2)

ax.set_xlabel('Date')
ax.set_ylabel('Total Weekly Sales ($ millions)')
ax.set_title('Total Weekly Sales Across All 45 Stores Over Time')
ax.legend(['Sales', 'Holiday weeks (red bands)'], loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_sales_over_time.png'))
plt.show()

**Observation:** There are clear annual spikes in late November / December (Thanksgiving + Christmas), with the peak week being **Dec 24, 2010** at $80.9M total. Sales also dip in late January. The pattern repeats each year, showing strong seasonality.

### 5.3 Average sales per store (sorted)

A horizontal bar chart showing each store's average weekly sales, sorted from lowest to highest.

In [ ]:
# Figure 3: Average sales per store, sorted
avg_by_store = df.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))
# Color gradient from red (low) to green (high)
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(avg_by_store)))
ax.barh(range(len(avg_by_store)), avg_by_store.values, color=colors)
ax.set_yticks(range(len(avg_by_store)))
ax.set_yticklabels([f'Store {s}' for s in avg_by_store.index], fontsize=8)
ax.set_xlabel('Average Weekly Sales ($)')
ax.set_title('Average Weekly Sales by Store (sorted)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_avg_sales_per_store.png'))
plt.show()

**Observation:** There is a huge variation across stores — Store 20 averages $2.1M/week while Store 33 averages only $260K/week, an ~8× difference. This means the `Store` feature will be very important for the model. The top 5 stores contribute a disproportionate share of total sales.

### 5.4 Holiday vs non-holiday sales

Comparing sales during holiday weeks vs regular weeks.

In [ ]:
# Figure 4: Holiday vs non-holiday sales
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Box plot comparison
holiday_data = [df[df['Holiday_Flag'] == 0]['Weekly_Sales'],
                df[df['Holiday_Flag'] == 1]['Weekly_Sales']]
bp = axes[0].boxplot(holiday_data, tick_labels=['Non-Holiday', 'Holiday'],
                     patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('salmon')
axes[0].set_ylabel('Weekly Sales ($)')
axes[0].set_title('Sales Distribution: Holiday vs Non-Holiday')

# Bar chart of averages
means = df.groupby('Holiday_Flag')['Weekly_Sales'].mean()
bars = axes[1].bar(['Non-Holiday', 'Holiday'], means.values,
                   color=['lightblue', 'salmon'], edgecolor='gray')
axes[1].set_ylabel('Average Weekly Sales ($)')
axes[1].set_title('Average Sales: Holiday vs Non-Holiday')
for bar, val in zip(bars, means.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,
                f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_holiday_vs_nonholiday.png'))
plt.show()

**Observation:** Holiday weeks have ~7.8% higher average sales ($1,122,888 vs $1,041,256). The difference is moderate, not dramatic — because `Holiday_Flag` covers Super Bowl, Labor Day, Thanksgiving, and Christmas, and some of those don't cause a big sales spike for every store.

### 5.5 Monthly / Week-of-year seasonality

Looking for seasonal patterns — which months and weeks of the year have the highest sales.

In [ ]:
# Add time features for the seasonality plots
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter'] = df['Date'].dt.quarter

# Figure 5: Monthly and weekly seasonality
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Monthly average sales
monthly_avg = df.groupby('Month')['Weekly_Sales'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
bars = axes[0].bar(range(1, 13), monthly_avg.values, color='steelblue',
                   edgecolor='white')
# Highlight November and December in red
bars[10].set_color('firebrick')  # November (index 10)
bars[11].set_color('firebrick')  # December (index 11)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(month_names)
axes[0].set_ylabel('Average Weekly Sales ($)')
axes[0].set_title('Average Weekly Sales by Month (Nov-Dec highlighted in red)')

# Week of year average sales
weekly_avg = df.groupby('WeekOfYear')['Weekly_Sales'].mean()
axes[1].plot(weekly_avg.index, weekly_avg.values, color='steelblue',
             linewidth=1.5, marker='o', markersize=3)
axes[1].set_xlabel('Week of Year')
axes[1].set_ylabel('Average Weekly Sales ($)')
axes[1].set_title('Average Weekly Sales by Week of Year')

# Label the peak week
peak_week = weekly_avg.idxmax()
axes[1].annotate(f'Peak: Week {peak_week}', xy=(peak_week, weekly_avg.max()),
                xytext=(peak_week - 8, weekly_avg.max() + 50000),
                arrowprops=dict(arrowstyle='->', color='red'),
                color='red', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_monthly_seasonality.png'))
plt.show()

**Observation:** December is the peak month and **Week 51** (the week before Christmas) is the single highest-selling week. Interestingly, the week before Christmas is **not flagged as a holiday** in the dataset (Holiday_Flag = 0), even though it has the highest sales. This is worth noting for model interpretation — Holiday_Flag alone doesn't capture the full holiday effect.

### 5.6 Correlation heatmap

Shows how strongly each pair of numeric columns is linearly related. Values close to +1 or -1 mean strong correlation; values near 0 mean weak or no linear relationship.

In [ ]:
# Figure 6: Correlation heatmap
numeric_cols = ['Weekly_Sales', 'Holiday_Flag', 'Temperature',
                'Fuel_Price', 'CPI', 'Unemployment', 'Store']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
# Only show the lower triangle (mirror is redundant)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, ax=ax, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_correlation_heatmap.png'))
plt.show()

**Observation:** All correlations with Weekly_Sales are weak. The strongest is Store number (-0.34), which makes sense because store numbering roughly correlates with store size. External features like Temperature (-0.06), CPI (-0.07), and Unemployment (-0.11) have very weak linear relationships with sales. This means tree-based models (which can capture non-linear patterns) should outperform Linear Regression.

### 5.7 Sales vs Temperature / CPI / Unemployment

Scatter plots showing how weekly sales relate to each external feature.

In [ ]:
# Figure 7: Scatter plots — Sales vs external features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scatter_features = ['Temperature', 'CPI', 'Unemployment']

for i, feat in enumerate(scatter_features):
    axes[i].scatter(df[feat], df['Weekly_Sales'], alpha=0.2, s=10,
                   color='steelblue')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Weekly Sales ($)')
    axes[i].set_title(f'Weekly Sales vs {feat}')
    
    # Show the correlation coefficient on the plot
    r = df[feat].corr(df['Weekly_Sales'])
    axes[i].text(0.05, 0.95, f'r = {r:.3f}', transform=axes[i].transAxes,
                fontsize=12, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Weekly Sales vs External Features (weak relationships)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_sales_vs_features.png'))
plt.show()

**Observation:** All three scatter plots show cloud-like patterns with no clear trend — confirming the weak correlations from the heatmap. Temperature (r=-0.064), CPI (r=-0.073), and Unemployment (r=-0.106) have almost no linear relationship with sales. The wide vertical spread at every temperature/CPI/unemployment value shows that **store identity matters far more** than these external factors.

---
## 6. Feature Engineering (Task 1.6)

We add time-based features from the Date column that the model will use:
- **Year** — captures long-term trends
- **Month** — captures monthly seasonality
- **WeekOfYear** — captures weekly seasonality (more precise than month)
- **Quarter** — captures seasonal patterns

The raw `Date` column will be dropped from model input later (in notebook 02).

In [ ]:
# We already added these features above for the seasonality plots.
# Let's verify they're in the dataframe:
print(f"Columns: {list(df.columns)}")
print(f"\nShape: {df.shape}")
print(f"\nSample of new features:")
print(df[['Date', 'Year', 'Month', 'WeekOfYear', 'Quarter']].head(10))

In [ ]:
# Save the feature-engineered dataframe as cleaned.csv
# This overwrites the earlier cleaned.csv to include the new columns
df.to_csv('../data/cleaned.csv', index=False)
print(f"Saved updated data to data/cleaned.csv ({df.shape[0]} rows, {df.shape[1]} columns)")
print(f"Columns: {list(df.columns)}")

---
## Summary

**Day 1 is complete.** Here's what we found:

1. The dataset is clean: no missing values, no duplicates, no invalid values.
2. Weekly sales vary hugely across stores ($210K – $3.8M), driven by store size.
3. Strong seasonality: November–December spike, peak in Week 51 (pre-Christmas).
4. Holiday flag gives ~7.8% higher average sales, but the week before Christmas isn't flagged.
5. External features (Temperature, CPI, Unemployment) have weak correlations with sales.
6. Store identity will likely be the most important predictor.

**Next (Day 2):** Build and evaluate 3 models — Linear Regression, Decision Tree, Random Forest.